## Part 1 — Setup

In [1]:
import asyncio
import csv
import json
import re
import shutil
import sys
import tempfile
from collections import Counter
from concurrent.futures import ThreadPoolExecutor
from contextlib import contextmanager
from importlib.metadata import version
from pathlib import Path

from IPython.display import Markdown, display
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

def run_async(coro):
    def worker():
        loop = asyncio.ProactorEventLoop() if sys.platform == 'win32' else asyncio.new_event_loop()
        asyncio.set_event_loop(loop)
        try:
            return loop.run_until_complete(coro)
        finally:
            loop.run_until_complete(loop.shutdown_asyncgens())
            loop.run_until_complete(loop.shutdown_default_executor())
            loop.close()
    with ThreadPoolExecutor(max_workers=1) as pool:
        return pool.submit(worker).result(timeout=180)

PROJECT_DIR = Path.cwd().resolve()
if not (PROJECT_DIR / 'project_mcp_server.py').is_file():
    PROJECT_DIR = PROJECT_DIR / 'mcp_project'
SERVER_FILE = PROJECT_DIR / 'project_mcp_server.py'
assert SERVER_FILE.is_file(), 'Open this notebook in mcp_project or its parent folder.'
KB_FILE = PROJECT_DIR / 'project_cve_kb.md'
EXPECTED = json.loads((PROJECT_DIR / 'expected_cves.json').read_text(encoding='utf8'))
FINAL_KB = KB_FILE.read_text(encoding='utf8')
FINAL_BYTES = KB_FILE.read_bytes()
TOOLS = ('lookup_cve', 'lookup_cve_v2')
BUCKETS = ('grounded', 'blind_spot', 'wrong_version', 'true_gap')
STARTER_HOSTS = ['10.0.0.5', '10.0.0.8', '10.0.0.12', '10.0.0.20']
MY_HOST = '10.0.0.30'

def parameters(server_file=SERVER_FILE):
    return StdioServerParameters(command=sys.executable, args=[str(server_file)])

def text_content(result):
    if getattr(result, 'isError', False):
        raise RuntimeError('MCP tool error: ' + str(result.content))
    return '\n'.join(item.text for item in result.content if hasattr(item, 'text'))

def sections(text):
    # Same heading convention as the starter. Bodies have no extra # characters.
    return {block.strip().splitlines()[0]: block.strip()
            for block in text.split('#') if block.strip()}

ADDED_TITLES = {'nginx 1.18.0', 'PostgreSQL 9.6.24', 'nginx 1.24.0', 'Apache httpd 2.4.50'}
CUSTOM_TITLES = {'nginx 1.24.0', 'Apache httpd 2.4.50'}
def without_titles(kb, excluded):
    return '\n\n'.join('# ' + body for title, body in sections(kb).items()
                       if title not in excluded) + '\n'
BASE_KB = without_titles(FINAL_KB, ADDED_TITLES)
PRACTICE_KB = without_titles(FINAL_KB, CUSTOM_TITLES)

@contextmanager
def snapshot(kb_text):
    # Each test starts the real server against a temporary KB copy.
    with tempfile.TemporaryDirectory(prefix='mcp_lab_') as folder:
        folder = Path(folder)
        server_copy = folder / SERVER_FILE.name
        shutil.copyfile(SERVER_FILE, server_copy)
        (folder / KB_FILE.name).write_text(kb_text, encoding='utf8')
        yield server_copy

print('Python:', sys.version.split()[0], '| MCP:', version('mcp'))
print('Server:', SERVER_FILE.name)
print('KB sections: baseline', len(sections(BASE_KB)),
      '| practice additions', len(sections(PRACTICE_KB)), '| final', len(sections(FINAL_KB)))

Python: 3.12.3 | MCP: 1.30.0
Server: project_mcp_server.py
KB sections: baseline 12 | practice additions 14 | final 16


## Part 2 — Connect and discover tools

TODO 3 is complete below. `stdio_client` starts the server using the notebook's Python interpreter. `initialize()` sets up the MCP session, and `list_tools()` asks the server what it offers. The tools and their descriptions come from the server, not from a list invented by the client.

In [2]:
async def discover(server_file=SERVER_FILE):
    with tempfile.TemporaryFile(mode='w+') as errlog:
        async with stdio_client(parameters(server_file), errlog=errlog) as (read, write):
            async with ClientSession(read, write) as session:
                initialized = await session.initialize()  # TODO 3
                offered = await session.list_tools()      # TODO 3
                return initialized.serverInfo.name, offered.tools

server_name, advertised_tools = run_async(discover())
print('Connected over stdio to:', server_name)
for tool in advertised_tools:
    print(tool.name, '—', (tool.description or '').splitlines()[0])
assert len(advertised_tools) == 5

Connected over stdio to: cyber-recon-project
port_scan — Read canned nmap-style banners for one known simulated lab IP.
list_targets — List the five fictional IPs accepted by this offline lab's host tools.
lookup_cve — Look up known vulnerabilities for a service or product name (e.g.
lookup_cve_v2 — Read local KB sections whose heading product and version match a banner.
check_port_policy — Compare a known simulated host's TCP ports with a fictional lab policy.


## Part 3 — Call a tool and read a resource

TODO 4 calls `port_scan` with a target argument. TODO 5 reads `kb://cve` with a resource URI. The resource is the whole file; a lookup tool returns selected sections.

In [3]:
async def scan_and_read():
    with tempfile.TemporaryFile(mode='w+') as errlog:
        async with stdio_client(parameters(), errlog=errlog) as (read, write):
            async with ClientSession(read, write) as session:
                await session.initialize()
                scan = await session.call_tool('port_scan', {'target': '10.0.0.5'})  # TODO 4
                resource = await session.read_resource('kb://cve')                 # TODO 5
                kb_text = '\n'.join(item.text for item in resource.contents if hasattr(item, 'text'))
                return text_content(scan), kb_text

scan_text, resource_text = run_async(scan_and_read())
print(scan_text)
print('\nRead kb://cve:', len(resource_text), 'characters;', len(sections(resource_text)), 'sections')
print('First resource heading:', next(iter(sections(resource_text))))
assert resource_text == FINAL_KB

Nmap scan report for 10.0.0.5
PORT     STATE SERVICE  VERSION
21/tcp   open  ftp      vsftpd 2.3.4
22/tcp   open  ssh      OpenSSH 7.2p2 Ubuntu
80/tcp   open  http     Apache httpd 2.4.49

Read kb://cve: 7756 characters; 16 sections
First resource heading: vsftpd 2.3.4


## Part 4 — Sweep the four starter hosts

This helper calls both lookup tools for every banner returned over MCP. It stops on a tool error instead of counting an error as a true gap. The policy tool is measured separately because an allowed port is not a CVE answer.

In [4]:
SERVICE_LINE = re.compile(r'^(\d+)/tcp\s+open\s+\S+\s+(.+)$', re.MULTILINE)

async def sweep(server_file, hosts, include_policy=False):
    rows, policies = [], {}
    with tempfile.TemporaryFile(mode='w+') as errlog:
        async with stdio_client(parameters(server_file), errlog=errlog) as (read, write):
            async with ClientSession(read, write) as session:
                await session.initialize()
                for host in hosts:
                    scan = text_content(await session.call_tool('port_scan', {'target': host}))
                    for port, banner in SERVICE_LINE.findall(scan):
                        for tool in TOOLS:
                            answer = text_content(await session.call_tool(tool, {'service': banner}))
                            rows.append({'host': host, 'port': int(port), 'service': banner,
                                         'tool': tool, 'answer': answer})
                    if include_policy:
                        result = await session.call_tool('check_port_policy', {'target': host})
                        policies[host] = json.loads(text_content(result))
    return rows, policies

with snapshot(BASE_KB) as server_copy:
    baseline_raw, _ = run_async(sweep(server_copy, STARTER_HOSTS))
print('Starter services:', len(baseline_raw) // 2, '| lookup answers:', len(baseline_raw))
assert len(baseline_raw) == 22

Starter services: 11 | lookup answers: 22


## Part 5 — Score the original knowledge base

The buckets are grounded, blind spot, wrong version, and true gap. I compare complete section titles rather than finding a word somewhere in the answer. A wrong extra section makes the answer wrong-version even when the expected section is also included. Two genuinely missing services stay in the true-gap bucket.

The 12-section snapshot includes one scope correction: MySQL 5.5.40 is later than the fix for CVE-2012-2122. Its expected course section stays `MySQL 5.5`, but retrieving that section is not proof that this banner is vulnerable.

In [5]:
def expected_map(final=False, custom=False):
    field = 'final_expected' if final else 'expected'
    stage = 'custom' if custom else 'starter'
    return {(row['host'], row['service']): row[field]
            for row in EXPECTED['services'] if row['stage'] == stage}

def returned_titles(answer):
    if answer.startswith('No known CVEs'):
        return []
    return [block.splitlines()[0].strip() for block in answer.strip().split('\n\n') if block.strip()]

def classify(answer, expected_title):
    found = returned_titles(answer)
    if not found:
        return 'true_gap' if expected_title is None else 'blind_spot'
    # Any extra or different heading is wrong, even if the expected one is present.
    return 'grounded' if expected_title is not None and found == [expected_title] else 'wrong_version'

def score(rows, truth, phase):
    scored = []
    for row in rows:
        wanted = truth[(row['host'], row['service'])]
        scored.append(dict(row, phase=phase, row_type='lookup', expected=wanted or '',
                           returned_sections=' | '.join(returned_titles(row['answer'])),
                           bucket=classify(row['answer'], wanted)))
    return scored

def counts(rows, tool):
    tally = Counter(row['bucket'] for row in rows if row['tool'] == tool)
    return tuple(tally[bucket] for bucket in BUCKETS)

def show_counts(rows, title):
    lines = ['### ' + title, '| Tool | Grounded | Blind spot | Wrong version | True gap |',
             '| --- | ---: | ---: | ---: | ---: |']
    for tool in TOOLS:
        lines.append('| ' + tool + ' | ' + ' | '.join(map(str, counts(rows, tool))) + ' |')
    display(Markdown('\n'.join(lines)))

baseline = score(baseline_raw, expected_map(), 'starter_baseline')
show_counts(baseline, 'Part 5 — original coverage')
assert counts(baseline, 'lookup_cve_v2') == (9, 0, 0, 2)
for row in baseline:
    if row['tool'] == 'lookup_cve' and row['bucket'] in ('blind_spot', 'wrong_version'):
        print(row['bucket'], '|', row['service'], '|', row['returned_sections'] or 'no section')

### Part 5 — original coverage
| Tool | Grounded | Blind spot | Wrong version | True gap |
| --- | ---: | ---: | ---: | ---: |
| lookup_cve | 6 | 2 | 1 | 2 |
| lookup_cve_v2 | 9 | 0 | 0 | 2 |

blind_spot | Linux telnetd | no section
wrong_version | Apache httpd 2.2.8 (OpenSSL 1.0.1) | Apache httpd 2.4.49 | OpenSSL 1.0.1
blind_spot | Redis key-value store 4.0.9 | no section


## Part 6 — Add missing nginx and PostgreSQL knowledge

The matching code is unchanged here. I add `nginx 1.18.0` for CVE-2021-23017 and `PostgreSQL 9.6.24` for unsupported-version risk (CWE-1104). The PostgreSQL note also explains that 9.6.24 **fixed** CVE-2021-23214. It would be wrong to call that CVE an active finding for this version.

The starter's two null expectations describe the old KB. This stage uses `final_expected` for those two services so the new coverage is scored fairly. The result should be 11 of 11 expected sections from v2.

In [6]:
with snapshot(PRACTICE_KB) as server_copy:
    practice_raw, _ = run_async(sweep(server_copy, STARTER_HOSTS))
practice = score(practice_raw, expected_map(final=True), 'starter_added_knowledge')
show_counts(practice, 'Part 6 — same code, more knowledge')
assert counts(practice, 'lookup_cve_v2') == (11, 0, 0, 0)
print('v2: 9 covered services + 2 true gaps became 11 grounded section matches.')

### Part 6 — same code, more knowledge
| Tool | Grounded | Blind spot | Wrong version | True gap |
| --- | ---: | ---: | ---: | ---: |
| lookup_cve | 8 | 2 | 1 | 0 |
| lookup_cve_v2 | 11 | 0 | 0 | 0 |

v2: 9 covered services + 2 true gaps became 11 grounded section matches.


## Part 7 — Audit the advertised contract

I would rewrite the original `lookup_cve` description before offering it to a model because its name sounds more reliable than its loose matching really is. I leave the function untouched for this comparison. I already changed the v2 description to explain substring matching, versionless headings, and the limits of “No known CVEs.” I also changed `port_scan` so an unknown target is an error, not a reassuring empty scan.

The new policy tool names its inputs, outputs, fixed policy, and limits. I would refuse to expose an unrestricted shell or a tool that changes firewall rules without a separate approval boundary. A descriptive docstring does not enforce permissions, and a server process by itself is not a sandbox.

In [7]:
async def audit_contract():
    with tempfile.TemporaryFile(mode='w+') as errlog:
        async with stdio_client(parameters(), errlog=errlog) as (read, write):
            async with ClientSession(read, write) as session:
                await session.initialize()
                tools = (await session.list_tools()).tools
                resources = (await session.list_resources()).resources
                rejected = await session.call_tool('check_port_policy', {'target': '203.0.113.99'})
                return tools, resources, rejected

audit_tools, audit_resources, rejected = run_async(audit_contract())
print('Resource URIs:', [str(r.uri) for r in audit_resources])
print('Unknown target returned tool error:', rejected.isError)
print('Error message:', '\n'.join(c.text for c in rejected.content if hasattr(c, 'text')))
assert rejected.isError
print('No real scan was attempted.')

Resource URIs: ['kb://cve']
Unknown target returned tool error: True
Error message: Error executing tool check_port_policy: Unknown simulated target; call list_targets first.
No real scan was attempted.


## Part 8A — My host and my tool

My fictional host is `10.0.0.30`. Its three banners are OpenSSH 7.2p2, nginx 1.24.0, and Apache httpd 2.4.50. OpenSSH was covered before my additions; the other two exact versions were missing. The nginx and Apache entries cite real advisories with the conditions that affect whether they apply.

My tool, `check_port_policy`, allows ports 22 and 443 under an invented lab policy. It marks other open ports for review and never changes anything. The next cell connects again and prints the description and input schema actually advertised by the server.

In [8]:
async def my_discovery():
    with tempfile.TemporaryFile(mode='w+') as errlog:
        async with stdio_client(parameters(), errlog=errlog) as (read, write):
            async with ClientSession(read, write) as session:
                hello = await session.initialize()
                tools = (await session.list_tools()).tools
                custom = next(t for t in tools if t.name == 'check_port_policy')
                targets = text_content(await session.call_tool('list_targets', {}))
                return hello, custom, targets

hello, custom_tool, targets = run_async(my_discovery())
print('Server started and initialized:', hello.serverInfo.name)
print('Protocol version:', hello.protocolVersion)
print(targets)
print('\nNEW TOOL:', custom_tool.name)
print(custom_tool.description)
print('\nINPUT SCHEMA:')
print(json.dumps(custom_tool.inputSchema, indent=2))
assert custom_tool.inputSchema['required'] == ['target']

Server started and initialized: cyber-recon-project
Protocol version: 2025-11-25
Known simulated targets: 10.0.0.5, 10.0.0.8, 10.0.0.12, 10.0.0.20, 10.0.0.30

NEW TOOL: check_port_policy
Compare a known simulated host's TCP ports with a fictional lab policy.

    target must exactly match an IP returned by list_targets. The policy allows
    ports 22 and 443; every other open port is marked review_required. Returns
    target, policy_id, allowed_tcp_ports, per-service port/banner/status/reason,
    and summary counts. Unknown targets return a tool error.
    This read-only check uses canned banners. It sends no packets, changes no
    firewall rules, and does not determine whether a service is vulnerable.
    review_required means an analyst should review a lab-policy exception;
    allowed does not mean secure. The policy is an example, not a standard.
    

INPUT SCHEMA:
{
  "properties": {
    "target": {
      "title": "Target",
      "type": "string"
    }
  },
  "required": [
   

## Part 8B — Live call to my policy tool

This is a real MCP call to the separate server process. Port 22 should be allowed. Ports 80 and 8080 should require review. “Allowed” only means the port fits this simple policy.

In [9]:
async def my_tool_call():
    with tempfile.TemporaryFile(mode='w+') as errlog:
        async with stdio_client(parameters(), errlog=errlog) as (read, write):
            async with ClientSession(read, write) as session:
                await session.initialize()
                scan = text_content(await session.call_tool('port_scan', {'target': MY_HOST}))
                policy = json.loads(text_content(await session.call_tool('check_port_policy', {'target': MY_HOST})))
                advisory = text_content(await session.call_tool('lookup_cve_v2', {'service': 'nginx 1.24.0'}))
                return scan, policy, advisory

my_scan, my_policy, my_advisory = run_async(my_tool_call())
print(my_scan)
print('\nPOLICY RESULT:')
for item in my_policy['services']:
    print(item['port'], item['status'], '—', item['banner'])
print('Counts:', my_policy['summary'])
print('\nLOOKUP RESULT:')
print(my_advisory)
assert my_policy['summary'] == {'allowed': 1, 'review_required': 2}

Nmap scan report for 10.0.0.30
PORT     STATE SERVICE  VERSION
22/tcp   open  ssh      OpenSSH 7.2p2 Ubuntu
80/tcp   open  http     nginx 1.24.0
8080/tcp open  http     Apache httpd 2.4.50

POLICY RESULT:
22 allowed — OpenSSH 7.2p2 Ubuntu
80 review_required — nginx 1.24.0
8080 review_required — Apache httpd 2.4.50
Counts: {'allowed': 1, 'review_required': 2}

LOOKUP RESULT:
nginx 1.24.0
CVE-2024-7347 is a buffer overread in ngx_http_mp4_module. The upstream affected range is 1.5.13 through 1.27.0, including 1.24.0; fixes include 1.26.2 and 1.27.1. The module must be built and configured for MP4 processing, and the attacker must be able to make the server process a crafted media file. A banner alone does not establish those conditions. Check the build and configuration, then use a supported, fixed package. Source: https://mailman.nginx.org/pipermail/nginx-announce/2024/UUOCLLONPR6244YQYU65PO5LB7JDYCWM.html


## Part 8C — Measure my host before and after adding knowledge

I run both lookup tools and the policy tool against the same three banners. The “before” copy already has the two Part 6 additions, but not my nginx 1.24.0 or Apache 2.4.50 sections. Then I use the complete precise KB. The policy check gets its own expected port statuses rather than being forced into the CVE buckets.

In [10]:
all_mine = []
policy_rows = []

def record_policy(policy, phase):
    expected_ports = EXPECTED['policy']['expected_by_port']
    output = []
    for item in policy['services']:
        wanted = expected_ports[str(item['port'])]
        output.append({'phase': phase, 'row_type': 'policy', 'host': MY_HOST,
                       'port': item['port'], 'service': item['banner'], 'tool': 'check_port_policy',
                       'expected_status': wanted, 'actual_status': item['status'],
                       'policy_correct': item['status'] == wanted, 'answer': item['reason']})
    assert all(row['policy_correct'] for row in output)
    return output

for phase, kb, final in [('before_custom_knowledge', PRACTICE_KB, False),
                         ('precise', FINAL_KB, True)]:
    with snapshot(kb) as server_copy:
        raw, policies = run_async(sweep(server_copy, [MY_HOST], include_policy=True))
    measured = score(raw, expected_map(final=final, custom=True), phase)
    all_mine.extend(measured)
    policy_rows.extend(record_policy(policies[MY_HOST], phase))
    show_counts(measured, phase)

precise = [r for r in all_mine if r['phase'] == 'precise']
assert counts(precise, 'lookup_cve_v2') == (3, 0, 0, 0)
print('Policy checks passed for all three ports in both runs.')

### before_custom_knowledge
| Tool | Grounded | Blind spot | Wrong version | True gap |
| --- | ---: | ---: | ---: | ---: |
| lookup_cve | 1 | 0 | 1 | 1 |
| lookup_cve_v2 | 1 | 0 | 0 | 2 |

### precise
| Tool | Grounded | Blind spot | Wrong version | True gap |
| --- | ---: | ---: | ---: | ---: |
| lookup_cve | 2 | 0 | 1 | 0 |
| lookup_cve_v2 | 3 | 0 | 0 | 0 |

Policy checks passed for all three ports in both runs.


## Part 8D — Deliberately drop a version and rerun

I change exactly one heading in a temporary KB copy: `# nginx 1.18.0` becomes `# nginx`. The advisory text and all banners stay the same. The v2 rule accepts versionless headings, so it now returns the old resolver advisory for nginx 1.24.0 as well as the correct MP4 advisory. The older resolver issue was fixed before 1.24.0, so that extra advisory is a wrong-version answer.

This is a controlled mistake for the experiment. The submitted knowledge base keeps the precise heading.

In [11]:
assert FINAL_KB.count('# nginx 1.18.0\n') == 1
VAGUE_KB = FINAL_KB.replace('# nginx 1.18.0\n', '# nginx\n', 1)
print('Changed heading: # nginx 1.18.0  ->  # nginx')
print('Only the heading changed:', VAGUE_KB.replace('# nginx\n', '# nginx 1.18.0\n', 1) == FINAL_KB)

with snapshot(VAGUE_KB) as server_copy:
    vague_raw, vague_policies = run_async(sweep(server_copy, [MY_HOST], include_policy=True))
vague = score(vague_raw, expected_map(final=True, custom=True), 'vague_heading')
all_mine.extend(vague)
policy_rows.extend(record_policy(vague_policies[MY_HOST], 'vague_heading'))
show_counts(precise, 'Before — precise headings')
show_counts(vague, 'After — version removed from the older nginx heading')
assert counts(vague, 'lookup_cve_v2') == (2, 0, 1, 0)

example = next(r for r in vague if r['tool'] == 'lookup_cve_v2' and r['service'] == 'nginx 1.24.0')
print('\nExpected:', example['expected'])
print('Returned:', example['returned_sections'])
print('Bucket:', example['bucket'])
print('\nActual tool answer:\n' + example['answer'])
assert KB_FILE.read_bytes() == FINAL_BYTES
print('\nFinal project KB still has its precise heading.')

Changed heading: # nginx 1.18.0  ->  # nginx
Only the heading changed: True


### Before — precise headings
| Tool | Grounded | Blind spot | Wrong version | True gap |
| --- | ---: | ---: | ---: | ---: |
| lookup_cve | 2 | 0 | 1 | 0 |
| lookup_cve_v2 | 3 | 0 | 0 | 0 |

### After — version removed from the older nginx heading
| Tool | Grounded | Blind spot | Wrong version | True gap |
| --- | ---: | ---: | ---: | ---: |
| lookup_cve | 1 | 0 | 2 | 0 |
| lookup_cve_v2 | 2 | 0 | 1 | 0 |


Expected: nginx 1.24.0
Returned: nginx | nginx 1.24.0
Bucket: wrong_version

Actual tool answer:
nginx
CVE-2021-23017 is a one-byte memory overwrite in the nginx DNS resolver. The upstream affected range is 0.6.18 through 1.20.0, which includes 1.18.0; fixed releases include 1.20.1 and 1.21.0. Exploitation depends on use of the resolver directive and an attacker able to forge a DNS response. Review configuration and vendor backports before treating a banner as vulnerable. Update to a supported, fixed package. Source: https://nginx.org/en/security_advisories.html

nginx 1.24.0
CVE-2024-7347 is a buffer overread in ngx_http_mp4_module. The upstream affected range is 1.5.13 through 1.27.0, including 1.24.0; fixes include 1.26.2 and 1.27.1. The module must be built and configured for MP4 processing, and the attacker must be able to make the server process a crafted media file. A banner alone does not establish those conditions. Check the build and configuration, then use a supported, fixe

## Part 8E — Save the results and check the final files

There are 18 lookup rows: 3 services × 2 tools × 3 KB stages. Nine additional rows record the separate policy checks. All four CVE buckets are present as table columns even when a count is zero. Part 5 demonstrates blind spots; my custom test mainly demonstrates true gaps and wrong-version answers.

In [12]:
fields = ['phase', 'row_type', 'host', 'port', 'service', 'tool', 'expected',
          'returned_sections', 'bucket', 'expected_status', 'actual_status',
          'policy_correct', 'answer']
csv_path = PROJECT_DIR / 'results_mine.csv'
with csv_path.open('w', encoding='utf8', newline='') as handle:
    writer = csv.DictWriter(handle, fieldnames=fields)
    writer.writeheader()
    writer.writerows(all_mine + policy_rows)

assert len(all_mine) == 18 and len(policy_rows) == 9
assert len(list(csv.DictReader(csv_path.open(encoding='utf8')))) == 27
assert KB_FILE.read_bytes() == FINAL_BYTES
print('Saved:', csv_path.name, '| 18 lookup rows + 9 policy rows = 27 rows')
for phase in ('before_custom_knowledge', 'precise', 'vague_heading'):
    stage_rows = [r for r in all_mine if r['phase'] == phase]
    show_counts(stage_rows, phase)
print('Custom policy: 22 allowed; 80 and 8080 review_required in every stage.')
print('Done. Save this notebook with all outputs showing before committing.')

Saved: results_mine.csv | 18 lookup rows + 9 policy rows = 27 rows


### before_custom_knowledge
| Tool | Grounded | Blind spot | Wrong version | True gap |
| --- | ---: | ---: | ---: | ---: |
| lookup_cve | 1 | 0 | 1 | 1 |
| lookup_cve_v2 | 1 | 0 | 0 | 2 |

### precise
| Tool | Grounded | Blind spot | Wrong version | True gap |
| --- | ---: | ---: | ---: | ---: |
| lookup_cve | 2 | 0 | 1 | 0 |
| lookup_cve_v2 | 3 | 0 | 0 | 0 |

### vague_heading
| Tool | Grounded | Blind spot | Wrong version | True gap |
| --- | ---: | ---: | ---: | ---: |
| lookup_cve | 1 | 0 | 2 | 0 |
| lookup_cve_v2 | 2 | 0 | 1 | 0 |

Custom policy: 22 allowed; 80 and 8080 review_required in every stage.
Done. Save this notebook with all outputs showing before committing.
